**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Bland-Altman Plots — DM vs DL-NoiseFree vs DL-Noisy  (SNR = 50)

Replaces the scatter (true vs pred) view with **Bland-Altman** to show
per-method bias and limits of agreement against ground truth.

For each method × parameter we plot:
- **x-axis**: mean of true and predicted = (true + pred) / 2
- **y-axis**: difference = pred − true
- **solid line** = mean bias
- **dashed lines** = bias ± 1.96·SD (limits of agreement)

All three methods are evaluated on the **same SNR=50 test set**.

## 1. Imports

In [ ]:
import os, time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DM       = '#E65100'   # deep orange — DM
C_DL_NF    = '#2E7D32'   # deep green  — DL noise-free
C_DL_NOISY = '#1565C0'   # deep blue   — DL noisy/triple

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('CPU only')
print(f'PyTorch {torch.__version__}')

## 2. Configuration — **edit paths here**

In [ ]:
CONFIG = {
    # ── Data ──────────────────────────────────────────────────────────────
    'noisefree_sig_path' : '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'param_path'         : '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'echotimes_path'     : '../echotimes.mat',
    'dict_key'           : 'Dico40_save',
    'param_key'          : 'par_save',

    # ── Model checkpoints ────────────────────────────────────────────────
    'ckpt_nf'    : './results/triple_regime_nf_results_v1/models/triple_nf_best.pt',
    'ckpt_noisy' : './results/triple_regime_results_v1/models/triple_regime_best.pt',

    # ── Output ────────────────────────────────────────────────────────────
    'output_dir' : './results/bland_altman_snr50',

    # ── Parameter space ───────────────────────────────────────────────────
    'param_mins'  : np.array([0.0,    0.0025,  1.0e-6,  0.050]),
    'param_maxs'  : np.array([1.0,    0.15,   25.0e-6,  0.200]),

    # ── GESFIDE geometry ─────────────────────────────────────────────────
    'n_fid'   : 14,
    'n_rephas': 16,
    'n_postse': 10,

    # ── Triple-regime feature scaling (must match training) ───────────────
    'R2starA_min':  2.0,   'R2starA_max': 55.0,
    'R2starB_min': -30.0,  'R2starB_max': 22.0,
    'R2starC_min':  2.0,   'R2starC_max': 55.0,

    # ── Test set ─────────────────────────────────────────────────────────
    'test_snr'   : 50,
    'n_test'     : 100_000,
    'n_scatter'  : 15_000,    # density-subsample for plotting only

    # ── DM ────────────────────────────────────────────────────────────────
    'dm_subsample': 200_000,
}

SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']
os.makedirs(CONFIG['output_dir'], exist_ok=True)
FIG_DIR = os.path.join(CONFIG['output_dir'], 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

PARAM_NAMES = ['SO₂', 'CBV',  'R',   'T2']
PARAM_UNITS = ['(%)',     '(%)',  '(µm)', '(ms)']
PARAM_SCALE = [100,        100,    1e6,    1000]
PARAM_LABELS = [f'{n} {u}' for n, u in zip(PARAM_NAMES, PARAM_UNITS)]

print(f'SE_ECHO = {SE_ECHO}')
print(f'Test SNR: {CONFIG["test_snr"]}')

## 3. Utilities

In [ ]:
def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    return signals[valid], params[valid]

def add_rician_noise(sig_nf, snr, rng=None):
    if rng is None: rng = np.random.default_rng(0)
    S0    = np.abs(sig_nf[:, 0:1])
    sigma = S0 / snr
    nr    = rng.normal(0, 1, sig_nf.shape).astype(np.float32) * sigma
    ni    = rng.normal(0, 1, sig_nf.shape).astype(np.float32) * sigma
    return np.sqrt((sig_nf + nr)**2 + ni**2).astype(np.float32)

def batched_predict(model, x_np, batch_size=4096, dev=device):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(x_np[i:i+batch_size], dtype=torch.float32).to(dev).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

def save_fig(fig, name):
    for ext in ['pdf', 'png']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext=='png' else None)
    print(f'  Saved: {name}')

print('Utilities ready.')

## 4. Echo times

In [ ]:
et_mat       = sio.loadmat(CONFIG['echotimes_path'])
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0

T_A     = echo_times_s[:CONFIG['n_fid']]
T_B     = echo_times_s[CONFIG['n_fid']:SE_ECHO]
T_C     = echo_times_s[SE_ECHO:]
T_SE_S  = echo_times_s[SE_ECHO - 1]
T_C_rel = T_C - T_SE_S
print(f'Echo times loaded.  SE at {T_SE_S*1e3:.1f} ms')

## 5. Triple-regime features

In [ ]:
def ols_slope(t_vec, sig_mat):
    log_s = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t     = t_vec.astype(np.float64)
    t_c   = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()

def compute_triple_regime_features(sig_raw, config):
    n_fid   = config['n_fid']
    se_echo = n_fid + config['n_rephas']
    r2_lb   = 1.0 / config['param_maxs'][3]
    slope_A = ols_slope(T_A,     sig_raw[:, :n_fid])
    slope_B = ols_slope(T_B,     sig_raw[:, n_fid:se_echo])
    slope_C = ols_slope(T_C_rel, sig_raw[:, se_echo:])
    R2starA = np.maximum(-slope_A, r2_lb).astype(np.float32)
    R2starB = (-slope_B).astype(np.float32)
    R2starC = np.maximum(-slope_C, r2_lb).astype(np.float32)
    return R2starA, R2starB, R2starC

def scale_triple_features(R2starA, R2starB, R2starC, config):
    def sc(x, lo, hi): return np.clip((x - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)
    return (sc(R2starA, config['R2starA_min'], config['R2starA_max']),
            sc(R2starB, config['R2starB_min'], config['R2starB_max']),
            sc(R2starC, config['R2starC_min'], config['R2starC_max']))

def build_43dim_input(sig_raw, config):
    R2A, R2B, R2C = compute_triple_regime_features(sig_raw, config)
    fA, fB, fC    = scale_triple_features(R2A, R2B, R2C, config)
    sig_norm       = euclidean_norm(sig_raw)
    return np.concatenate([sig_norm, fA[:, None], fB[:, None], fC[:, None]], axis=1)

print('Feature functions ready.')

## 6. Model architecture

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)

class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, cond_in=3, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2 * feature_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        b = torch.zeros(2 * feature_dim); b[:feature_dim] = 1.0
        self.net[-1].bias.data.copy_(b)
        self.feature_dim = feature_dim
    def forward(self, x, cond):
        p = self.net(cond)
        return p[:, :self.feature_dim] * x + p[:, self.feature_dim:]

class TripleRegimeModel(nn.Module):
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32,  7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128,3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        ch = film_cond_hidden
        self.fc1   = nn.Linear(1280, 512); self.bn1 = nn.BatchNorm1d(512)
        self.film1 = FiLMLayer(512, 3, ch)
        self.fc2   = nn.Linear(512,  256); self.bn2 = nn.BatchNorm1d(256)
        self.film2 = FiLMLayer(256, 3, ch)
        self.fc3   = nn.Linear(256,  128); self.bn3 = nn.BatchNorm1d(128)
        self.film3 = FiLMLayer(128, 3, ch)
        self.fc_out  = nn.Linear(128, n_outputs)
        self.out_act = Clamp01()
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()
    def forward(self, x):
        echo = x[:, :40].unsqueeze(1)
        cond = x[:, 40:43]
        c = self.conv(echo).flatten(1)
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), cond))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), cond))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), cond))))
        return self.out_act(self.fc_out(h))

_m = TripleRegimeModel().to(device)
print(f'TripleRegimeModel OK — {sum(p.numel() for p in _m.parameters()):,} params')
del _m

## 7. Build SNR=50 test set

In [ ]:
print('Loading noise-free dictionary...')
sig_nf  = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
par_all = load_mat(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]

sig_nf, par_all = filter_param_range(sig_nf, par_all, CONFIG['param_mins'], CONFIG['param_maxs'])
sig_nf, par_all = clean_data(sig_nf, par_all)
print(f'Clean dictionary: {len(sig_nf):,} samples')

n_test = min(CONFIG['n_test'], len(sig_nf))
rng = np.random.default_rng(42)
idx_test = rng.choice(len(sig_nf), n_test, replace=False)
sig_test_nf = sig_nf[idx_test]
par_test    = par_all[idx_test]

SNR_TEST = CONFIG['test_snr']
sig_test_noisy = add_rician_noise(sig_test_nf, SNR_TEST, rng=np.random.default_rng(99))

par_test_phys = par_test.copy()
for i, sc in enumerate(PARAM_SCALE):
    par_test_phys[:, i] = par_test[:, i] * sc

print(f'\nTest set: {n_test:,} samples @ SNR={SNR_TEST}')

## 8. DM — GPU inner-product matching

In [ ]:
print('Running Dictionary Matching (DM) on GPU...')
t0 = time.time()

dm_sub = CONFIG.get('dm_subsample')
if dm_sub is not None and dm_sub < len(sig_nf):
    rng_dm = np.random.default_rng(7)
    idx_dm = rng_dm.choice(len(sig_nf), dm_sub, replace=False)
    dm_dict_sig = sig_nf[idx_dm]
    dm_dict_par = par_all[idx_dm]
else:
    dm_dict_sig = sig_nf
    dm_dict_par = par_all

dm_dict_norm_gpu = torch.tensor(euclidean_norm(dm_dict_sig),
                                 dtype=torch.float32).to(device)
test_norm_gpu    = torch.tensor(euclidean_norm(sig_test_noisy),
                                 dtype=torch.float32).to(device)

CHUNK = 4096
dm_pred_idx = np.empty(n_test, dtype=np.int32)
with torch.no_grad():
    for start in range(0, n_test, CHUNK):
        end = min(start + CHUNK, n_test)
        ip  = test_norm_gpu[start:end] @ dm_dict_norm_gpu.T
        dm_pred_idx[start:end] = ip.argmax(dim=1).cpu().numpy()

dm_pred_par  = dm_dict_par[dm_pred_idx]
dm_pred_phys = dm_pred_par.copy()
for i, sc in enumerate(PARAM_SCALE):
    dm_pred_phys[:, i] = dm_pred_par[:, i] * sc
print(f'  DM done in {time.time()-t0:.1f}s')

## 9. DL — Noise-Free model

In [ ]:
print('Loading DL Noise-Free model...')
model_nf = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
state_nf = torch.load(CONFIG['ckpt_nf'], map_location=device)
model_nf.load_state_dict(state_nf)
model_nf.eval()

x_test_input = build_43dim_input(sig_test_noisy, CONFIG)
nf_pred_scaled = batched_predict(model_nf, x_test_input)
nf_pred_raw    = params_inverse(nf_pred_scaled,
                                CONFIG['param_mins'][:4],
                                CONFIG['param_maxs'][:4])
nf_pred_phys = nf_pred_raw.copy()
for i, sc in enumerate(PARAM_SCALE):
    nf_pred_phys[:, i] = nf_pred_raw[:, i] * sc
print('  DL-NF inference done')

## 10. DL — Noisy (Triple A+B+C Mixed-SNR) model

In [ ]:
print('Loading DL Noisy model...')
model_noisy = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
state_noisy = torch.load(CONFIG['ckpt_noisy'], map_location=device)
model_noisy.load_state_dict(state_noisy)
model_noisy.eval()

noisy_pred_scaled = batched_predict(model_noisy, x_test_input)
noisy_pred_raw    = params_inverse(noisy_pred_scaled,
                                   CONFIG['param_mins'][:4],
                                   CONFIG['param_maxs'][:4])
noisy_pred_phys = noisy_pred_raw.copy()
for i, sc in enumerate(PARAM_SCALE):
    noisy_pred_phys[:, i] = noisy_pred_raw[:, i] * sc
print('  DL-Noisy inference done')

## 11. Bland-Altman plotting helper

In [ ]:
def make_bland_altman_row(axes, true_phys, pred_phys, colour, row_label,
                            n_scatter=15_000):
    """
    Fill one row of 4 Bland-Altman subplots (one per parameter).
    """
    rng_sc = np.random.default_rng(0)
    idx_sc = rng_sc.choice(len(true_phys),
                           min(n_scatter, len(true_phys)), replace=False)

    for ci, (ax, pname, punit) in enumerate(zip(axes, PARAM_NAMES, PARAM_UNITS)):
        t = true_phys[:, ci]
        p = pred_phys[:, ci]
        v = np.isfinite(t) & np.isfinite(p)
        t, p = t[v], p[v]

        # Bland-Altman quantities
        mean_tp = (t + p) / 2.0
        diff    = p - t
        bias    = float(np.mean(diff))
        sd      = float(np.std(diff, ddof=1))
        loa_lo  = bias - 1.96 * sd
        loa_hi  = bias + 1.96 * sd

        # Scatter (random subset for readability)
        sub = idx_sc[idx_sc < len(t)]
        ax.scatter(mean_tp[sub], diff[sub], s=1.8, alpha=0.10,
                   color=colour, rasterized=True)

        # x-range based on full data, y-range based on diff distribution
        x_lo, x_hi = np.nanpercentile(mean_tp, [0.5, 99.5])
        y_max = max(abs(loa_lo), abs(loa_hi)) * 1.4
        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(-y_max, y_max)

        # Zero reference
        ax.axhline(0, color='#888888', lw=0.7, ls=':', zorder=1)

        # Bias and limits of agreement
        ax.axhline(bias,   color='#D32F2F', lw=1.6, zorder=3)
        ax.axhline(loa_lo, color='#212121', lw=1.2, ls='--', zorder=3)
        ax.axhline(loa_hi, color='#212121', lw=1.2, ls='--', zorder=3)

        # Annotate values on the right side
        x_text = x_lo + 0.95 * (x_hi - x_lo)
        # ax.text(x_text, bias,   f'  bias = {bias:+.2f}',
        #         ha='right', va='bottom', fontsize=7,
        #         color='#E65100', fontweight='bold')
        # ax.text(x_text, loa_hi, f'  +1.96 SD = {loa_hi:+.2f}',
        #         ha='right', va='bottom', fontsize=6.5, color='#1565C0')
        # ax.text(x_text, loa_lo, f'  -1.96 SD = {loa_lo:+.2f}',
        #         ha='right', va='top',    fontsize=6.5, color='#1565C0')
        ax.text(x_text, bias,   f'  bias = {bias:+.2f}',
                ha='right', va='bottom', fontsize=12,
                color='#D32F2F', fontweight='bold')                          # red
        # ax.text(x_text, bias, f'  bias = {bias:+.2f}',
        #     ha='right', va='bottom', fontsize=7,
        #     color='#D32F2F', fontweight='bold',
        #     bbox=dict(facecolor='white', edgecolor='none',
        #               alpha=0.85, pad=1.5))
        ax.text(x_text, loa_hi, f'  +1.96 SD = {loa_hi:+.2f}',
                ha='right', va='bottom', fontsize=12, color='#212121')      # black
        ax.text(x_text, loa_lo, f'  -1.96 SD = {loa_lo:+.2f}',
                ha='right', va='top',    fontsize=12, color='#212121')      # black

        ax.set_xlabel(f'Mean of true & pred', fontsize=12)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(labelsize=8)

    axes[0].set_ylabel(f'{row_label}\nPred − True', fontsize=12)

print('Bland-Altman helper ready.')

## 12. Main figure — 3 rows × 4 columns

In [ ]:
N_SCATTER = CONFIG['n_scatter']

fig, axes = plt.subplots(
    3, 4,
    figsize=(13, 9),
    gridspec_kw={'wspace': 0.40, 'hspace': 0.55}
)

# ── Row 0: DM ─────────────────────────────────────────────────────────────
make_bland_altman_row(axes[0], par_test_phys, dm_pred_phys,
                      colour=C_DL_NOISY, row_label='DM', n_scatter=N_SCATTER)

# ── Row 1: DL Noise-Free ──────────────────────────────────────────────────
make_bland_altman_row(axes[1], par_test_phys, nf_pred_phys,
                      colour=C_DL_NOISY, row_label='DL\nNoise-Free',
                      n_scatter=N_SCATTER)

# ── Row 2: DL Noisy (Triple A+B+C) ────────────────────────────────────────
make_bland_altman_row(axes[2], par_test_phys, noisy_pred_phys,
                      colour=C_DL_NOISY, row_label='DL\nNoisy',
                      n_scatter=N_SCATTER)

# ── Column titles ────────────────────────────────────────────────────────
for ci, label in enumerate(PARAM_LABELS):
    axes[0, ci].set_title(label, fontsize=12, fontweight='bold')

fig.suptitle(
    f'Bland-Altman — Predicted vs Ground Truth (SNR = {CONFIG["test_snr"]})\n'
    f'DM  |  DL Noise-Free  |  DL Noisy (Triple A+B+C)',
    fontsize=12, y=1.02
)

save_fig(fig, f'bland_altman_snr{CONFIG["test_snr"]}_DM_DLnf_DLnoisy')
plt.show()
print('Done.')

## 13. Summary metrics table

In [ ]:
import pandas as pd

def ba_metrics(true_phys, pred_phys, method):
    rows = []
    for ci, (pname, punit) in enumerate(zip(PARAM_NAMES, PARAM_UNITS)):
        t = true_phys[:, ci]; p = pred_phys[:, ci]
        v = np.isfinite(t) & np.isfinite(p)
        diff = p[v] - t[v]
        bias = float(np.mean(diff))
        sd   = float(np.std(diff, ddof=1))
        rows.append({
            'method'   : method,
            'parameter': pname,
            'unit'     : punit,
            'bias'     : bias,
            'sd_diff'  : sd,
            'loa_lower': bias - 1.96 * sd,
            'loa_upper': bias + 1.96 * sd,
            'n'        : int(v.sum()),
        })
    return rows

table = []
table += ba_metrics(par_test_phys, dm_pred_phys,    'DM')
table += ba_metrics(par_test_phys, nf_pred_phys,    'DL-NoiseFree')
table += ba_metrics(par_test_phys, noisy_pred_phys, 'DL-Noisy')

df = pd.DataFrame(table)
df.to_csv(os.path.join(CONFIG['output_dir'],
                      f'bland_altman_metrics_snr{CONFIG["test_snr"]}.csv'),
          index=False)

# Pretty print
print(f'\n{"="*78}')
print(f'Bland-Altman summary — SNR = {CONFIG["test_snr"]}')
print(f'{"="*78}')
print(f'{"Method":<14} {"Param":<6} {"Bias":>10} {"SD":>8} {"LoA Lower":>12} {"LoA Upper":>12}')
print('-'*78)
for r in table:
    print(f'{r["method"]:<14} {r["parameter"]:<6} '
          f'{r["bias"]:>+9.3f}{r["unit"]:<2} '
          f'{r["sd_diff"]:>7.3f}  '
          f'{r["loa_lower"]:>+11.3f}  {r["loa_upper"]:>+11.3f}')
print('='*78)
df